In [60]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [61]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.set_printoptions(suppress=True)

In [62]:
from simulators import NestedModelFamily, ContextManager
from simulators.benchmarks import DDM, RDM, CDM
from adapters import Adapter

# Metas

In [63]:
ddm_intrinsics = ["v", "a", "tau", "s_v", "s_tau", "decay"]

# Priors

In [64]:
ddm_priors = {
    "v":     {"intercept": lambda: np.random.gamma(3.0, 0.8),
              "slope":     lambda: np.random.normal(0.0, 3.0)},
    "a":     {"intercept": lambda: np.random.gamma(10.0, 0.3),
              "slope":     lambda: np.random.normal(0.0, 1.0)},
    "tau":   {"intercept": lambda: np.random.gamma(3.0, 0.2),
              "slope":     lambda: 0.0},
    "s_v":   {"intercept": lambda: np.random.gamma(1.0, 0.2),
              "slope":     lambda: 0.0},
    "s_tau": {"intercept": lambda: np.random.uniform(0.0, 0.4),
              "slope":     lambda: 0.0},
    "decay": {"intercept": lambda: np.random.gamma(1.0, 0.4),
              "slope":     lambda: 0.0},
}

# Context Manager

In [65]:
context_manager = ContextManager()

# Model Family

In [66]:
model_family = NestedModelFamily(
    name="DDM",
    model=DDM(),
    context_manager=context_manager,
    prior_fun=ddm_priors,
    intrinsic_params=ddm_intrinsics,
)

In [70]:
samples = model_family.batch_sample(
    batch_size=3,
    mask_randomizer_kwargs=dict(
        free_intrinsics={"v", "a", "tau", "s_v", "decay"},
        fixed_intrinsics={"s_tau"}
    ),
    min_num_obs=20,
    max_num_obs=500,
    flatten_param_outputs=False
)

ValueError: could not broadcast input array from shape (72,) into shape (12,6)

In [23]:
samples["param_masks"].shape

(3, 126)

In [24]:
samples["param_matrices"].shape

(3, 126)

In [25]:
samples["regressor_masks"].shape

(3, 21)

In [26]:
samples

{'model_names': ['DDM', 'DDM', 'DDM'],
 'design_configs': [{'u_0': ['a', 'tau', 'decay'],
   'u_1': ['a', 'tau', 's_v'],
   'u_2': ['v'],
   'u_3': ['v', 'decay'],
   'u_4': ['v', 'a'],
   'u_5': ['v', 's_v'],
   'u_6': ['v', 'a', 'tau', 'decay']},
  {'u_0': ['v', 'tau'], 'u_1': ['a', 'decay']},
  {'u_0': ['v', 'a', 'tau', 'decay'],
   'u_1': ['tau', 's_v'],
   'u_2': ['tau', 'decay'],
   'u_3': ['v', 'tau', 'decay'],
   'u_4': ['s_v', 'decay']}],
 'design_matrices': array([[[0.60018108, 0.        , 0.        , ..., 1.        ,
          0.        , 0.        ],
         [0.84245615, 0.        , 0.        , ..., 1.        ,
          0.        , 0.        ],
         [0.56615871, 0.        , 0.        , ..., 0.        ,
          1.        , 0.        ],
         ...,
         [0.32810461, 0.        , 0.        , ..., 0.        ,
          1.        , 0.        ],
         [0.18319067, 0.        , 0.        , ..., 0.        ,
          1.        , 0.        ],
         [0.06486823, 0. 

# Adapter

In [27]:
adapter = Adapter()

In [28]:
design_matrices = adapter.convert_dtype(samples["design_matrices"], dtype=np.float32)
param_masks = adapter.convert_dtype(samples["param_masks"], dtype=np.float32)
rts = adapter.convert_dtype(samples["sim_data"]["rts"], dtype=np.float32)
choices = adapter.convert_dtype(samples["sim_data"]["choices"], dtype=np.float32)

In [29]:
batch_size, num_obs, num_cols = design_matrices.shape
print(batch_size, num_obs, num_cols)

3 344 21


In [30]:
y_rts_col = adapter.atleast_2d(rts, orientation="col")                 # (N, 1)
y_ch_col  = adapter.atleast_2d(rts,  orientation="col")

In [31]:
sim_data = adapter.concatenate([y_rts_col, y_ch_col], axis=1, dtype=np.float32, pad=False)

# RDM

In [32]:
rdm_priors = {
    "v":      {"intercept": lambda: np.random.gamma(3.0, 0.8),
               "slope":     lambda: np.random.normal(0.0, 3.0)},
    "a":      {"intercept": lambda: np.random.gamma(10.0, 0.3),
               "slope":     lambda: np.random.normal(0.0, 1.0)},
    "tau":    {"intercept": lambda: np.random.gamma(3.0, 0.2),
               "slope":     lambda: np.random.normal(0.0, 0.2)},
    "decay":  {"intercept": lambda: np.random.gamma(1.0, 0.4),
               "slope":     lambda: np.random.normal(0.0, 0.2)},
}

In [34]:
family = NestedModelFamily(
    name="RDM",
    model=RDM(),
    context_manager=context_manager,
    prior_fun=rdm_priors,
    intrinsic_params=ddm_intrinsics,
)

In [35]:
num_alternatives = np.random.randint(2, 4, size=1)  # number of alternatives

context = {
    "correct_idx": lambda n: np.random.randint(0, num_alternatives, size=n),
    "num_alternatives": num_alternatives,
}

In [36]:
samples = family.batch_sample(
    batch_size=3,
    mask_randomizer_kwargs=dict(
        free_intrinsics={"v", "a", "tau"},
        fixed_intrinsics={"decay"}
    ),
    min_num_obs=20,
    max_num_obs=500,
    context=context,
)

In [37]:
samples

{'model_names': ['RDM', 'RDM', 'RDM'],
 'design_configs': [{'u_0': ['v', 'a'],
   'u_1': ['a', 'tau'],
   'u_2': ['v'],
   'u_3': ['a'],
   'u_4': ['v', 'a'],
   'u_5': ['a', 'tau']},
  {'u_0': ['a'], 'u_1': ['v', 'tau'], 'u_2': ['v', 'a']},
  {'u_0': ['v', 'tau'],
   'u_1': [],
   'u_2': ['v', 'a'],
   'u_3': ['v', 'a', 'tau'],
   'u_4': ['v', 'a', 'tau'],
   'u_5': ['a'],
   'u_6': ['v', 'a'],
   'u_7': ['v'],
   'u_8': ['v', 'tau'],
   'u_9': ['v', 'a', 'tau']}],
 'design_matrices': array([[[0.83727419, 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.7341004 , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.78540182, 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         ...,
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
      

# CDM

In [38]:
cdm_intrinsics = ["v_x", "v_y", "a", "tau", "decay"]

In [39]:
cdm_priors = {
    "v_x":   {"intercept": lambda: np.random.normal(0.0, 1.0),
              "slope":     lambda: np.random.normal(0.0, 0.5)},
    "v_y":   {"intercept": lambda: np.random.normal(0.0, 1.0),
              "slope":     lambda: np.random.normal(0.0, 0.5)},
    "a":     {"intercept": lambda: np.random.gamma(10.0, 0.3),
              "slope":     lambda: np.random.normal(0.0, 0.5)},
    "tau":   {"intercept": lambda: np.random.gamma(3.0, 0.2),
              "slope":     lambda: np.random.normal(0.0, 0.1)},
    "decay": {"intercept": lambda: np.random.gamma(1.0, 0.4),
              "slope":     lambda: np.random.normal(0.0, 0.1)},
}

In [40]:
family = NestedModelFamily(
    name="CDM",
    model=CDM(),
    context_manager=context_manager,
    prior_fun=cdm_priors,
    intrinsic_params=cdm_intrinsics,
)

In [41]:
out = family.sample(
    design_config=None,
    num_obs=200,
    num_regressors=3,
    max_num_regressors=6,
    max_num_categories=4,   # dummy blocks up to 3 cols per regressor
    keep_intercept=True,
    discrete_prob=0.5,      # mix of continuous and dummy regressors
)

In [42]:
print("Keys:", out.keys())
print("design_matrix:", out["design_matrix"].shape)
print("param_mask:", out["param_mask"].shape)
print("param_matrix:", out["param_matrix"].shape)
print("sim rts/choices:", out["sim_trials"]["rts"].shape, out["sim_trials"]["choices"].shape)

Keys: dict_keys(['model_name', 'design_config', 'design_matrix', 'param_mask', 'param_matrix', 'sim_trials', 'discrete_mask', 'regressor_mask', 'max_num_regressors', 'keep_intercept'])
design_matrix: (200, 10)
param_mask: (50,)
param_matrix: (50,)
sim rts/choices: (200,) (200,)


In [43]:
batch = family.batch_sample(
    batch_size=4,
    num_obs=None,                 # randomized per item
    num_regressors=None,          # randomized per item
    min_num_obs=100,
    max_num_obs=250,
    min_num_regressors=0,
    max_num_regressors=5,
    max_num_categories=4,
    keep_intercept=True,
    discrete_prob=0.6,
)

In [44]:
print("Batched design_matrices:", batch["design_matrices"].shape)
print("Batched sim rts:", batch["sim_data"]["rts"].shape)
print("Batched sim choices:", batch["sim_data"]["choices"].shape)
print("num_obs per item:", batch["num_obs"].reshape(-1))
print("num_regressors per item:", batch["num_regressors"].reshape(-1))

Batched design_matrices: (4, 189, 16)
Batched sim rts: (4, 189)
Batched sim choices: (4, 189)
num_obs per item: [153. 172. 189. 175.]
num_regressors per item: [5. 5. 5. 5.]
